# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：**Steve-cza**  
第5天专题（A/B/C/D/E）：**A**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "Steve-cza"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [ ]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


In [ ]:
# 业务问题与图表选择理由
business_questions = {
    "category_bar": "不同生命周期阶段（TenureGroup）用户的流失率有何差异？",
    "behavior_scatter": "用户订单次数（OrderCount）与返现金额（CashbackAmount）之间是否存在关联？流失用户与未流失用户的行为分布有何不同？",
    "ordered_line": "随着生命周期阶段的递进，流失率呈现怎样的有序变化趋势？",
    "composition_chart": "全体用户中，投诉用户（Complain=1）与未投诉用户（Complain=0）各自占比多少？",
}

chart_reasons = {
    "category_bar": "TenureGroup是离散类别（共5个阶段），目标是比较各类别的流失率差异，因此选择柱状图；添加标签时同时显示比例和样本量。",
    "behavior_scatter": "OrderCount和CashbackAmount均为连续数值变量，一个点代表一个用户，适合用散点图观察整体分布和群体差异；按Churn着色可区分流失状态。",
    "ordered_line": "TenureGroup是有序阶段（新用户→24个月以上），比较各阶段流失率变化趋势，折线图能清晰展示上升或下降的方向性。",
    "composition_chart": "Complain只有2个类别（未投诉/投诉），合计为全体用户，适合用饼图或环形图展示整体构成比例。",
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")


## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

category_field = "TenureGroup"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
category_summary[category_field] = pd.Categorical(
    category_summary[category_field], categories=TENURE_ORDER, ordered=True
)
category_summary = category_summary.sort_values(category_field).reset_index(drop=True)

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)


In [ ]:
# 绘制柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

bars = ax_bar.bar(
    range(len(category_summary)),
    category_summary["流失率"],
    color=["#2196F3", "#FF9800", "#4CAF50", "#9C27B0", "#607D8B"],
    width=0.6, edgecolor="white",
)
ax_bar.set_xticks(range(len(category_summary)))
ax_bar.set_xticklabels(category_summary[category_field], fontsize=11)
ax_bar.set_title("不同生命周期阶段用户流失率对比", fontsize=14, fontweight="bold")
ax_bar.set_xlabel("生命周期阶段", fontsize=12)
ax_bar.set_ylabel("流失率", fontsize=12)
ax_bar.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_bar.set_ylim(0, max(category_summary["流失率"]) * 1.25)

for bar_obj, (_, row) in zip(bars, category_summary.iterrows()):
    ax_bar.text(
        bar_obj.get_x() + bar_obj.get_width() / 2,
        bar_obj.get_height() + 0.008,
        f"{row["流失率"]:.1%}\n(n={int(row["用户数"])})",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
    )
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)

bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))


### 柱状图结论

- 观察：随着用户生命周期从新用户向长期用户过渡，流失率呈现显著的下降趋势。新用户流失率超过50%，而24个月以上用户的流失率为0%。
- 证据：新用户（508人）流失率53.5%，0-6个月用户（1642人）流失率25.9%，7-12个月用户（1584人）流失率9.8%，13-24个月用户（1467人）流失率6.5%，24个月以上用户（429人）流失率0%。各组样本量均充足。
- 边界：柱状图展示的是截面数据的阶段间差异，不能证明"使用时长直接导致留存"。可能存在自选择偏差——本身更活跃的用户倾向于使用更长时间。


## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

CHURN_COLORS = {0: "#4CAF50", 1: "#F44336"}
CHURN_LABELS = {0: "未流失", 1: "流失"}

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

for churn_val in [0, 1]:
    subset = df[df["Churn"] == churn_val]
    ax_scatter.scatter(
        subset[x_field], subset[y_field],
        c=CHURN_COLORS[churn_val],
        label=f"{CHURN_LABELS[churn_val]} (n={len(subset)})",
        alpha=0.4, s=20, edgecolors="none",
    )
ax_scatter.set_title("订单次数与返现金额关系（按流失状态着色）", fontsize=14, fontweight="bold")
ax_scatter.set_xlabel("订单次数 (OrderCount)", fontsize=12)
ax_scatter.set_ylabel("返现金额 (CashbackAmount)", fontsize=12)
ax_scatter.legend(fontsize=11, loc="upper right")
ax_scatter.spines["top"].set_visible(False)
ax_scatter.spines["right"].set_visible(False)

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))


### 散点图结论

- 观察：流失用户（红色）集中在散点图的左下区域（订单次数少、返现金额低），未流失用户（绿色）分布更广且整体偏向右上区域。
- 证据：流失用户相比未流失用户整体订单次数和返现金额更低。订单次数与返现金额之间存在一定的正相关趋势。未流失用户有948人，流失用户4682人。
- 边界：散点图只能展示两个变量之间的相关关系和群体分布差异，不能推导因果关系。例如不能断定"低返现导致流失"，两者可能受共同第三变量（如用户活跃度）影响。


## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
ordered_summary[ordered_field] = pd.Categorical(
    ordered_summary[ordered_field], categories=TENURE_ORDER, ordered=True
)
ordered_summary = ordered_summary.sort_values(ordered_field).reset_index(drop=True)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)


In [ ]:
# 绘制折线图（有序阶段比较）
fig_line, ax_line = plt.subplots(figsize=(10, 6))

ax_line.plot(
    range(len(ordered_summary)),
    ordered_summary["流失率"],
    marker="o", linewidth=2.5, markersize=10, color="#E91E63",
)
ax_line.set_xticks(range(len(ordered_summary)))
ax_line.set_xticklabels(ordered_summary[ordered_field], fontsize=11)
ax_line.set_title("用户生命周期阶段与流失率变化（有序阶段比较）", fontsize=14, fontweight="bold")
ax_line.set_xlabel("生命周期阶段（有序）", fontsize=12)
ax_line.set_ylabel("流失率", fontsize=12)
ax_line.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_line.set_ylim(0, max(ordered_summary["流失率"]) * 1.25)

for i, (_, row) in enumerate(ordered_summary.iterrows()):
    ax_line.annotate(
        f"{row["流失率"]:.1%}\n(n={int(row["用户数"])})",
        (i, row["流失率"]),
        textcoords="offset points", xytext=(0, 18),
        ha="center", fontsize=10, fontweight="bold",
    )
ax_line.spines["top"].set_visible(False)
ax_line.spines["right"].set_visible(False)

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))


### 折线图结论

- 观察：从新用户到24个月以上用户，流失率呈单调下降趋势。最大的下降幅度发生在新用户→0-6个月（从53.5%降至25.9%）。
- 证据：新用户（508人）流失率53.5% → 0-6个月（1642人）25.9% → 7-12个月（1584人）9.8% → 13-24个月（1467人）6.5% → 24个月以上（429人）0%。每个阶段的样本量均充足。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势。折线只反映不同用户群体在同一时点的流失率差异，不能预测同一批用户未来的流失率变化。


## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
composition_field = "Complain"
composition_summary = (
    df.groupby(composition_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"))
      .reset_index()
)
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()
composition_summary["标签"] = composition_summary[composition_field].map({0: "未投诉", 1: "投诉"})

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
import numpy as np
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)


In [ ]:
# 类别不超过5个，使用环形图
fig_composition, ax_composition = plt.subplots(figsize=(8, 8))

colors_comp = ["#4CAF50", "#F44336"]
wedges, texts, autotexts = ax_composition.pie(
    composition_summary["用户数"],
    labels=composition_summary["标签"],
    autopct=lambda p: f"{p:.1f}%\n({int(p*composition_summary["用户数"].sum()/100)}人)",
    startangle=90,
    colors=colors_comp,
    textprops={"fontsize": 12},
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight("bold")
ax_composition.set_title("用户投诉状态构成", fontsize=14, fontweight="bold", pad=20)

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))


### 构成图结论

- 观察：大多数用户（71.5%）没有投诉记录，投诉用户约占全体用户的四分之一（28.5%）。
- 证据：未投诉用户4026人（占71.5%），投诉用户1604人（占28.5%）。两者合计为全体5630名用户。
- 边界：环形图适合展示整体构成比例，但不便于比较各子组内部的其他指标（如投诉组内的流失率差异）。对于更细致的分析，需要配合分组柱状图或交叉表。


## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
CHURN_COLORS = {0: "#4CAF50", 1: "#F44336"}
CHURN_LABELS = {0: "未流失", 1: "流失"}
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# 准备各图数据
cat_data = (
    df.groupby("TenureGroup", observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
cat_data["TenureGroup"] = pd.Categorical(cat_data["TenureGroup"], categories=TENURE_ORDER, ordered=True)
cat_data = cat_data.sort_values("TenureGroup").reset_index(drop=True)

comp_data = (
    df.groupby("Complain", observed=True)
      .agg(用户数=("CustomerID", "nunique"))
      .reset_index()
)
comp_data["标签"] = comp_data["Complain"].map({0: "未投诉", 1: "投诉"})

fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# 左上：柱状图
ax = axes[0, 0]
bars = ax.bar(range(len(cat_data)), cat_data["流失率"],
    color=["#2196F3", "#FF9800", "#4CAF50", "#9C27B0", "#607D8B"],
    width=0.6, edgecolor="white")
ax.set_xticks(range(len(cat_data)))
ax.set_xticklabels(cat_data["TenureGroup"], fontsize=9)
ax.set_title("各阶段流失率", fontsize=12, fontweight="bold")
ax.set_ylabel("流失率")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
for b, (_, r) in zip(bars, cat_data.iterrows()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
        f"{r["流失率"]:.1%}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 右上：散点图
ax = axes[0, 1]
for cv in [0, 1]:
    sub = df[df["Churn"] == cv]
    ax.scatter(sub["OrderCount"], sub["CashbackAmount"],
        c=CHURN_COLORS[cv], label=CHURN_LABELS[cv], alpha=0.3, s=10, edgecolors="none")
ax.set_title("订单次数 vs 返现金额", fontsize=12, fontweight="bold")
ax.set_xlabel("订单次数")
ax.set_ylabel("返现金额")
ax.legend(fontsize=9, loc="upper right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 左下：折线图
ax = axes[1, 0]
ax.plot(range(len(cat_data)), cat_data["流失率"],
    marker="o", linewidth=2.5, markersize=8, color="#E91E63")
ax.set_xticks(range(len(cat_data)))
ax.set_xticklabels(cat_data["TenureGroup"], fontsize=9)
ax.set_title("阶段流失率变化", fontsize=12, fontweight="bold")
ax.set_xlabel("生命周期阶段")
ax.set_ylabel("流失率")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
for i, (_, r) in enumerate(cat_data.iterrows()):
    ax.annotate(f"{r["流失率"]:.1%}", (i, r["流失率"]),
        textcoords="offset points", xytext=(0, 12),
        ha="center", fontsize=8, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# 右下：构成柱状图
ax = axes[1, 1]
bars_comp = ax.bar(comp_data["标签"], comp_data["用户数"],
    color=["#4CAF50", "#F44336"], width=0.5, edgecolor="white")
ax.set_title("投诉状态分布", fontsize=12, fontweight="bold")
ax.set_ylabel("用户数")
total = comp_data["用户数"].sum()
for b, v in zip(bars_comp, comp_data["用户数"]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+20,
        f"{v}\n({v/total*100:.1f}%)", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.relative_to(ROOT))


## 综合发现与局限

1. 综合发现1：生命周期阶段与流失率显著负相关。新用户流失率高达53.5%，而24个月以上用户流失率为0%。随着用户在平台留存时间增长，流失风险持续降低，说明早期用户留存是降低整体流失率的关键。
2. 综合发现2：订单次数和返现金额与流失状态存在关联。流失用户整体订单次数偏低（集中在1-3次），返现金额也低于未流失用户，说明用户活跃度和价值贡献可能与流失风险密切相关。
3. 综合发现3：投诉用户占全体用户的28.5%（1604人），该比例值得关注。结合第5天分析，投诉用户中流失率较高，投诉行为与流失存在关联，但需进一步验证是否为因果关系。
4. 数据或方法局限：本数据为截面数据，缺少订单时间序列信息，无法计算复购率、留存率等动态指标。CashbackAmount是返现金额，不是销售额、收入或GMV。各组间的差异仅反映当前样本中的关联，不能直接推导因果关系。

注意：`CashbackAmount`是返现金额，不是销售额、收入或GMV。


## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
# 图表清单（5行）
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png",
     "business_question": "不同生命周期阶段的用户流失率有何差异？",
     "chart_type": "bar",
     "key_finding": "新用户流失率最高(37.4%)，24个月以上用户流失率最低(3.5%)，生命周期越长流失率越低",
     "limitation": "仅反映当前样本分布，不能证明使用时长直接导致留存"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png",
     "business_question": "订单次数与返现金额之间是否存在关联？流失用户与未流失用户的行为模式有何不同？",
     "chart_type": "scatter",
     "key_finding": "流失用户集中在订单次数少、返现金额低的区域；未流失用户分布更广、整体订单和返现更高",
     "limitation": "散点图展示相关关系，不能推导因果关系"},
    {"chart_id": "03", "file_name": "03_ordered_line.png",
     "business_question": "用户生命周期阶段的有序变化与流失率之间呈现什么趋势？",
     "chart_type": "line",
     "key_finding": "从新用户到24个月以上用户，流失率呈单调下降趋势，说明用户留存时间越长越稳定",
     "limitation": "这是有序阶段比较，不是时间序列趋势，不能预测未来流失率"},
    {"chart_id": "04", "file_name": "04_composition_chart.png",
     "business_question": "投诉用户在全体用户中占比多少？",
     "chart_type": "pie",
     "key_finding": "未投诉用户占71.5%(4026人)，投诉用户占28.5%(1604人)，大多数用户未投诉",
     "limitation": "饼图适合展示整体构成，不便于比较细粒度差异"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png",
     "business_question": "整体概览",
     "chart_type": "dashboard",
     "key_finding": "生命周期阶段与流失率显著负相关；订单返现与流失状态相关；投诉用户占比约28.5%",
     "limitation": "综合图仅概括核心发现，部分细节在独立图中更清晰"},
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)


In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")
